# Liquid Token Valuation: <Protocol> ($TICKER)

Target-price model, run like a listed equity. Bear / Base / Bull cases build revenue bottom-up through a generic driver tree: TAM -> market share -> **units** -> **monetization rate** -> gross revenue -> take rate -> net revenue -> EBITDA margin -> EBITDA. `units` is the protocol's traction quantity (a stock like active loans / AUM, or an annual flow like trading volume); the monetization rate is the fee take on it, so gross revenue = units x rate either way. Exit value = exit-year EBITDA x the exit FDV/EBITDA multiple; exit price = exit value / fully-diluted supply. Returns are entry/exit only: no interim cash flows (buybacks accrue via price). Fair value today = prob-weighted exit price discounted at the hurdle. All levels and multiples are stated on FDV and mcap.

**How to instantiate for a new deal:** copy this repo (or just this notebook + `src/`), fill in the DEAL CONFIG cell below, set the scenario drivers, Run All, then edit NARRATIVE / MANUAL and re-run the one-pager section. The shipped example values run Aave end-to-end so a fresh clone works out of the box.

Sections: (1) assumptions, (2) revenue build, (3) income statement, (4) entry/exit and scenarios, (4b) sensitivity grids, (4c) MOIC waterfall, comps, one-pager.

EBITDA is the bottom line and a loose UFCF proxy (tax, D&A, capex, NWC not modeled). Compute logic in `src/valuation/`; $ in millions. Data: DeFiLlama (units = `borrowed` or TVL per `UNITS_METRIC`; category TAM cached in `data/units_history.json` via the DeFiLlama MCP).

In [ ]:
# Auto-reload edited src/valuation modules without a kernel restart.
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('src')

from datetime import date
import pandas as pd
from valuation import marks, model, comps, display

# ==================================================================================
# DEAL CONFIG -- EDIT EVERYTHING IN THIS CELL (example values run Aave end-to-end)
# ==================================================================================
PROTOCOL_NAME = 'Aave'                # EDIT: display name
TICKER        = 'AAVE'                # EDIT: token ticker

# Data source
DEFILLAMA_SLUG        = 'aave'        # EDIT: DeFiLlama protocol slug
GECKO_ID              = 'aave'        # EDIT: CoinGecko id
ANNUALIZE_WINDOW_DAYS = 30            # trailing Nd x 365/N; 7 | 30 | 365
USE_LIVE_DATA         = True          # False -> most recent saved snapshot
FALLBACK_MARKS        = None          # optional dict of last-known-good marks (offline backstop)

# Units: the protocol's traction quantity behind the revenue build.
#   UNITS_METRIC: 'borrowed' (lending active loans) | 'tvl' | None (set marks_now['units'] yourself)
#   UNIT_LABEL:   how the deal names it (drives tables, the one-pager and the comps column)
UNITS_METRIC = 'borrowed'             # EDIT
UNIT_LABEL   = 'Active loans'         # EDIT: e.g. 'Active loans', 'AUM', 'Trading volume'
TAM_CATEGORY = 'Lending'              # EDIT: DeFiLlama category for the live TAM check; None to skip

# Market sizing
TAM_START = 25000.0                   # EDIT: addressable market ($mm), first forecast year

# Valuation knobs
HURDLE_RATE     = 0.40                # discount rate for fair value today
VALUATION_DATE  = date.today()        # EDIT: "today" (year 0); pin a date for reproducible runs
EXIT_YEAR       = VALUATION_DATE.year + 3   # EDIT: exit at EOY

# MOIC/IRR entry price. Candidates print in the Key Assumptions table.
#   'blended_pv' prob-weighted fair value (pw IRR = hurdle)
#   'base_pv'    base-case PV (base IRR = hurdle)
#   'market'     current market price
#   'manual'     ENTRY_PRICE below
ENTRY_BASIS = 'base_pv'               # EDIT
ENTRY_PRICE = None                    # used only when ENTRY_BASIS = 'manual'

# Comps (a deal judgement, not a default): token peers as (name, slug, gecko),
# equity peers as {group label: [tickers]}. Maple's set is left as a reference.
PEER_COMPS = [('Morpho', 'morpho', 'morpho'),          # EDIT
              ('Kamino', 'kamino', 'kamino'),
              ('Maple', 'maple-finance', 'syrup')]
EQUITY_COMPS = {                                       # EDIT
    'Crypto adjacent': ['COIN', 'CRCL'],
    # Maple's capital-light set, for reference:
    # 'Private credit / platforms': ['ARES', 'APO', 'OWL', 'BX', 'KKR', 'BAM', 'CG', 'HLNE', 'PGY'],
    # 'Crypto adjacent':            ['COIN', 'CRCL', 'FIGR'],
}

# Display labels: override the sector-neutral defaults with the deal's own words
# (keys = model row keys / scenario columns; see display.DEFAULT_LABELS).
LABELS = {                                             # EDIT (or leave {})
    'units': UNIT_LABEL,
    # Maple's lending example, for reference:
    # 'tam': 'Active Loans TAM', 'units': 'Maple Active Loans',
    # 'fee_rate': 'Gross Loan Yield', 'revenue': 'Maple Revenue',
    # 'share_target': 'Maple Market Share at Exit', 'fee_rate_exit': 'Gross Loan Yield at Exit',
}

HOLD_YRS = (date(EXIT_YEAR, 12, 31) - VALUATION_DATE).days / 365.25

assert ENTRY_BASIS in model.ENTRY_BASES, f'ENTRY_BASIS must be one of {model.ENTRY_BASES}'
assert ENTRY_BASIS != 'manual' or ENTRY_PRICE is not None, "ENTRY_BASIS='manual' needs ENTRY_PRICE"

## Current marks

Live DeFiLlama pull with per-field provenance; every pull snapshots to `data/marks_snapshot.json` (set `USE_LIVE_DATA = False` to run offline from it; a fresh clone with no snapshot needs one live run or a `FALLBACK_MARKS` dict). FDV = max supply x price, the supply metric behind every multiple (mcap = circulating x price alongside; the treasury/emissions fields feed the one-pager token-economics block).

In [ ]:
marks_now, prov, meta = marks.get_marks(
    USE_LIVE_DATA, DEFILLAMA_SLUG, GECKO_ID, ANNUALIZE_WINDOW_DAYS, HOLD_YRS,
    units_metric=UNITS_METRIC, fallback=FALLBACK_MARKS)
# UNITS_METRIC=None: set the units mark yourself before proceeding, e.g.
#   marks_now['units'] = <annualized volume, $mm>

PRICE_NOW = marks_now['price']
FDV_NOW   = marks.fdv_mm(marks_now)

live_n = sum(v == 'live' for v in prov.values())
print(f"Marks source: {meta['source']}  (asof {meta['asof']}, window {meta['window_days']}d)")
print(f"  {live_n}/{len(prov)} fields live")
_units_str = f"{UNIT_LABEL.lower()} ${marks_now['units']:,.0f}m  " if marks_now.get('units') else ''
print(f"  price ${marks_now['price']:.4f}  mcap ${marks_now['mcap']:,.1f}m  "
      f"{_units_str}fdv {marks_now['fdv_supply']:,.0f}m")
print(f"  revenue ${marks_now['revenue']:,.1f}m (annualized {ANNUALIZE_WINDOW_DAYS}d)   "
      f"fees ${marks_now['fees']:,.1f}m   holders-rev ${marks_now['holders_rev']:,.2f}m")
if marks_now.get('emissions_hold_mm') is not None:   # Pro emission endpoint; absent without a key
    print(f"  emissions over hold ({HOLD_YRS:.1f}yr): {marks_now['emissions_hold_mm']:,.2f}m tokens")
print()
print(f"FDV today           : ${FDV_NOW:,.1f}m   (mcap ${marks_now['mcap']:,.1f}m)")
print(f"FDV / revenue (ann) : {FDV_NOW / marks_now['revenue']:,.1f}x")
print(f"FDV / fees (ann)    : {FDV_NOW / marks_now['fees']:,.1f}x")

## Scenario drivers: Bear / Base / Bull

Probabilities sum to 1; add a case by extending the lists. Per case: TAM growth (CAGR after year 1), four drivers that ramp linearly from a common first-forecast-year start to the case exit target (market share, monetization rate, take rate, EBITDA margin), and `exit_pe`, the exit FDV/EBITDA multiple (hard-coded). Live values print below for reference.

In [ ]:
CASES = ['Bear', 'Base', 'Bull']
ACTIVE_CASE = 'Base'            # case shown in the statements; also the base-case reference

# Common first-forecast-year starts, same for every case; each ramps linearly to
# its per-case exit target. EDIT all four (live values print below for reference).
SHARE_START         = 0.30     # EDIT: market share (units / TAM)
FEE_RATE_START      = 0.05     # EDIT: monetization rate (fees / units, p.a.)
TAKE_RATE_START     = 0.15     # EDIT: take rate (net revenue / gross revenue)
EBITDA_MARGIN_START = 0.30     # EDIT: EBITDA margin

scenario = pd.DataFrame({                              # EDIT every row
    'prob':               [0.25,  0.50,  0.25],
    'mkt_growth':         [0.05,  0.10,  0.15],   # TAM CAGR, applied after year 1
    'share_target':       [0.30,  0.35,  0.40],   # market share at exit
    'fee_rate_exit':      [0.05,  0.055, 0.06],   # monetization rate at exit
    'take_rate_exit':     [0.15,  0.17,  0.20],   # take rate at exit
    'ebitda_margin_exit': [0.30,  0.40,  0.50],   # EBITDA margin at exit
    'exit_pe':            [20,    30,    40],      # exit FDV/EBITDA, hard-coded
}, index=CASES)
assert abs(scenario['prob'].sum() - 1.0) < 1e-9, 'Probabilities must sum to 100%'

# Live take rate for the reference print.
_take_rate_live = marks_now['revenue'] / marks_now['fees']

ctx = model.make_context(marks_now, VALUATION_DATE, EXIT_YEAR, TAM_START,
                         EBITDA_MARGIN_START, FEE_RATE_START, TAKE_RATE_START, SHARE_START)

# TAM assumption vs the live DeFiLlama category total (sanity check only).
_tam_live = marks.category_tam_now_mm(TAM_CATEGORY, UNITS_METRIC or 'tvl') if TAM_CATEGORY else None
if _tam_live:
    _gap = TAM_START - _tam_live
    print(f"TAM check: assumption ${TAM_START:,.0f}m  vs  live {TAM_CATEGORY}-category "
          f"${_tam_live:,.0f}m   ->   {_gap:+,.0f}m ({_gap / _tam_live:+.1%})")

_live_share = f"{ctx['share_now']:.1%}" if ctx['share_now'] else 'n/a'
_live_fee   = f"{ctx['fee_rate_now']:.1%}" if ctx['fee_rate_now'] else 'n/a'
print(f"Active case: {ACTIVE_CASE}   |   spot ${PRICE_NOW:.4f}   |   "
      f"forecast {ctx['year_labels'][0]}-{ctx['year_labels'][-1]} (exit EOY {EXIT_YEAR})")
print(f"Market share      {SHARE_START:.1%} start (live {_live_share}) "
      f"-> {scenario.loc[ACTIVE_CASE,'share_target']:.1%} exit (Base)")
print(f"Monetization rate {FEE_RATE_START:.1%} start (live {_live_fee}) "
      f"-> {scenario.loc[ACTIVE_CASE,'fee_rate_exit']:.1%} exit (Base)")
print(f"Take rate         {TAKE_RATE_START:.1%} start (live {_take_rate_live:.1%}) "
      f"-> {scenario.loc[ACTIVE_CASE,'take_rate_exit']:.1%} exit (Base)")
print(f"EBITDA margin     {EBITDA_MARGIN_START:.0%} start "
      f"-> {scenario.loc[ACTIVE_CASE,'ebitda_margin_exit']:.0%} exit (Base)")
print(f"Exit FDV/EBITDA (hard-coded): " + ", ".join(f"{c} {scenario.loc[c,'exit_pe']:.0f}x" for c in CASES))

## 1) Key assumptions

Global knobs, the entry-price candidates, and the by-scenario drivers.

In [ ]:
# Entry candidates are computable before the ENTRY_BASIS choice matters
# (value_case is entry-independent). The selected basis is marked.
_entries = model.entry_candidates(scenario, ctx, marks_now, HURDLE_RATE, ACTIVE_CASE)
_sel = lambda b: '  <-- ENTRY' if ENTRY_BASIS == b else ''

global_items = [
    ('Valuation Date',              VALUATION_DATE,   'raw'),
    ('Exit (EOY)',                  EXIT_YEAR,        'raw'),
    ('Hold to Exit',                ctx['tau_exit'],  'years'),
    ('Hurdle Rate',                 HURDLE_RATE,      'pct0'),
    ('Entry Basis (toggle)',        ENTRY_BASIS,      'raw'),
    ('Entry: Blended PV' + _sel('blended_pv'), _entries['blended_pv'], 'price'),
    ('Entry: Base-case PV' + _sel('base_pv'),  _entries['base_pv'],    'price'),
    ('Entry: Market Price' + _sel('market'),   _entries['market'],     'price'),
] + ([('Entry: Manual ENTRY_PRICE  <-- ENTRY', ENTRY_PRICE, 'price')] if ENTRY_BASIS == 'manual' else []) + [
    ('Spot Price',                  PRICE_NOW,        'price'),
    ('TAM (yr 1)',                  TAM_START,        'usd_mm0'),
    ('TAM (live)',                  _tam_live,        'usd_mm0'),
    ('Market Share (start)',        SHARE_START,      'pct1'),
    ('Market Share (live)',         ctx['share_now'], 'pct1'),
    ('Monetization Rate (start)',   FEE_RATE_START,   'pct1'),
    ('Monetization Rate (live)',    ctx['fee_rate_now'], 'pct1'),
    ('Take Rate (start)',           TAKE_RATE_START,  'pct1'),
    ('Take Rate (live)',            _take_rate_live,  'pct1'),
    ('EBITDA Margin (start)',       EBITDA_MARGIN_START, 'pct0'),
    ('Mcap Today',                  marks_now['mcap'], 'usd_mm'),
    ('FDV Today',                   FDV_NOW,          'usd_mm'),
]
print('GLOBAL ASSUMPTIONS')
print(display.global_assumptions_table(global_items).to_string())
print('\nBY-SCENARIO DRIVERS')
display.scenario_assumptions_table(scenario, labels=LABELS)

## 2) Revenue build

Active case, bottom-up: TAM (input level in year 1, then case CAGR) x market share = units; x monetization rate = gross revenue; x take rate = net revenue. Ramped drivers start at their first-forecast-year inputs, so year 1 is a pro-forma stub.

In [ ]:
proj = model.build_projection(ACTIVE_CASE, scenario, ctx)
print(f'Revenue build - {ACTIVE_CASE} case ($m; {ctx["year_labels"][0]} is a pro-forma stub)')
display.revenue_build_table(proj, labels=LABELS)

## 3) Income statement

Historical columns are DeFiLlama actuals, no model assumptions: the last three full fiscal years, YTD, TTM, and the trailing-window annualized run-rate (TAM / units context from `data/units_history.json`, filled from the DeFiLlama Pro categories history and the protocol's own series, hand-entered values kept; monetization rate = annualized gross revenue / average units). Projected columns are the model build. EBITDA is blank historically (protocol opex unobserved) and is the projected bottom line, a loose UFCF proxy. Tie-out lines confirm the DeFiLlama identity: Fees - supply-side = Revenue = protocol + holders.

In [ ]:
# Historical income from DeFiLlama actuals + the units-history cache. With DEFILLAMA_API_KEY the
# Pro financial statement adds the incentives / earnings lines and the cache's missing TAM / units
# averages are filled from the Pro categories history; hand-entered values stay.
_y0 = VALUATION_DATE.year
_full_years = [_y0 - 3, _y0 - 2, _y0 - 1]
inc = {y: marks.defillama_income(DEFILLAMA_SLUG, y, USE_LIVE_DATA) for y in _full_years + [_y0]}
uh = marks.ensure_units_history(DEFILLAMA_SLUG, TAM_CATEGORY, UNITS_METRIC, _full_years + [_y0], USE_LIVE_DATA)
ytd_days = max((date.today() - date(_y0, 1, 1)).days, 1)   # annualizes the YTD monetization rate

def _hist(label, year, ann_factor):
    d, c = inc.get(year), uh.get(str(year), {})
    if not d:
        return None
    return (label, {'fees': d['fees'], 'revenue': d['revenue'],
                    'tam': c.get('tam'), 'units': c.get('units'),
                    'ann_factor': ann_factor})

periods = [p for p in (
    [_hist(f'FY{y}', y, 1.0) for y in _full_years]
    + [_hist(f'YTD {_y0}', _y0, 365 / ytd_days)]
) if p]

# TTM: trailing-365d fees/revenue + trailing-12m-average units and TAM (the cache's 'TTM' row;
# TAM falls back to the live snapshot without a Pro key). Reused by the comps subject below.
ttm_inc = marks.defillama_income_ttm(DEFILLAMA_SLUG, USE_LIVE_DATA)
_uh_ser = marks.units_history_mm(DEFILLAMA_SLUG, UNITS_METRIC) if UNITS_METRIC else []
_cut = (pd.Timestamp.now('UTC') - pd.Timedelta(days=365)).timestamp()
_ttm_window = [v for ts, v in _uh_ser if ts >= _cut]
_ttm_units = (sum(_ttm_window) / len(_ttm_window)) if _ttm_window else marks_now.get('units')
if ttm_inc:
    periods.append(('TTM', {'fees': ttm_inc['fees'], 'revenue': ttm_inc['revenue'],
                            'tam': uh.get('TTM', {}).get('tam') or _tam_live,
                            'units': _ttm_units, 'ann_factor': 1.0}))
periods.append((f'T{ANNUALIZE_WINDOW_DAYS}D Ann.',
                {'fees': marks_now['fees'], 'revenue': marks_now['revenue'],
                 'tam': _tam_live, 'units': marks_now.get('units'), 'ann_factor': 1.0}))

hist_is = model.historical_income(periods)
print(f'Income statement: {ACTIVE_CASE} case ($m; historical = DeFiLlama actuals, projected = model build)')
for lbl, d in [(f'FY{y}', inc.get(y)) for y in _full_years] + [(f'YTD {_y0}', inc.get(_y0)), ('TTM', ttm_inc)]:
    if d:
        _extra = (f"  incentives {d['incentives']:.1f} -> earnings {d['earnings']:.1f}"
                  if d.get('earnings') is not None else '')
        print(f"  {lbl} DeFiLlama tie-out: Fees {d['fees']:.1f} - Supply-side {d['supply_side']:.1f} "
              f"= Revenue {d['revenue']:.2f}m  (= Protocol {d['protocol_rev']:.1f} + Holders {d['holders_rev']:.2f})"
              f"{_extra}  [{d.get('source', 'cache')}]")
display.income_statement(proj, hist=hist_is, labels=LABELS)

## 4) Entry / exit analysis and scenarios

Per case: exit value = exit-year EBITDA x `exit_pe`; exit price = exit value / FDV supply. No interim cash flows, no net-cash bridge. Fair value today = prob-weighted PV of the exit price at the hurdle (an underwriting number, not intrinsic value).

Entry price comes from the `ENTRY_BASIS` toggle (candidates in section 1). MOIC = exit price / entry; IRR = MOIC^(1/hold) - 1. By construction: a blended-PV entry gives pw IRR = hurdle, a base-case-PV entry gives base IRR = hurdle (asserted). The headline entry multiple (entry x FDV supply / current annualized EBITDA) is a display-only check on `exit_pe` vs the comps.

In [ ]:
val, summary = model.run_scenarios(scenario, ctx, marks_now, HURDLE_RATE, ACTIVE_CASE,
                                   ENTRY_BASIS, ENTRY_PRICE)
# by-construction checks per basis, plus assumptions-table candidates vs resolved values
if ENTRY_BASIS == 'blended_pv':
    assert abs(summary['pw_irr'] - HURDLE_RATE) < 1e-9
elif ENTRY_BASIS == 'base_pv':
    assert abs(val.loc[ACTIVE_CASE, 'irr'] - HURDLE_RATE) < 1e-9
assert abs(_entries['blended_pv'] - summary['fair_value_today']) < 1e-12
assert abs(_entries['base_pv'] - summary['base_pv_price']) < 1e-12
display.entry_exit_report(val, summary, scenario)

## 4b) MOIC sensitivity grids

Base-case MOIC, entry held fixed: TAM x exit share, and exit FDV/EBITDA x exit share. The run-value cell reproduces the reported MOIC (asserted). `model.moic_grid` accepts any two drivers (scenario columns or `model.CTX_DRIVERS` keys), so other pairs are one-line changes.

In [ ]:
# axes: any two drivers (scenario columns or model.CTX_DRIVERS keys).
# EDIT the axes so the run values (TAM_START, the case share_target / exit_pe) appear in them.
TAM_AXIS   = sorted({TAM_START * f for f in (0.6, 0.8, 1.0, 1.2, 1.4)})
SHARE_AXIS = sorted({scenario.loc[c, 'share_target'] for c in CASES} | {SHARE_START})
PE_AXIS    = sorted({scenario.loc[c, 'exit_pe'] for c in CASES} | {15, 25})
_entry = summary['returns_entry']

_g1 = model.moic_grid(ACTIVE_CASE, scenario, ctx, marks_now, HURDLE_RATE, _entry,
                      ('tam_start', TAM_AXIS), ('share_target', SHARE_AXIS))
_g2 = model.moic_grid(ACTIVE_CASE, scenario, ctx, marks_now, HURDLE_RATE, _entry,
                      ('exit_pe', PE_AXIS), ('share_target', SHARE_AXIS))

# the run-value cells must reproduce the reported case MOIC
_share_run = scenario.loc[ACTIVE_CASE, 'share_target']
assert abs(_g1.loc[TAM_START, _share_run] - val.loc[ACTIVE_CASE, 'moic']) < 1e-9
assert abs(_g2.loc[scenario.loc[ACTIVE_CASE, 'exit_pe'], _share_run] - val.loc[ACTIVE_CASE, 'moic']) < 1e-9

def _fmt_grid(g, row_fmt, col_fmt, row_name):
    out = pd.DataFrame([[f'{v:.1f}x' for v in row] for _, row in g.iterrows()],
                       index=[row_fmt(r) for r in g.index],
                       columns=[col_fmt(c) for c in g.columns])
    out.index.name = row_name
    return out

print(f'{ACTIVE_CASE}-case MOIC  |  cols = market share at exit  |  entry held at ${_entry:.4f}')
print('\n(i) Initial TAM x exit share')
print(_fmt_grid(_g1, lambda t: f'${t/1000:.0f}b', lambda s: f'{s*100:.1f}%', 'Initial TAM').to_string())
print('\n(ii) Exit FDV/EBITDA x exit share')
print(_fmt_grid(_g2, lambda p: f'{p:.0f}x', lambda s: f'{s*100:.1f}%', 'Exit multiple').to_string())

## 4c) MOIC waterfall

Exact multiplicative bridge from 1.0x at entry to each case MOIC, one bar per assumption (order-independent, no residual, asserted): MOIC = TAM growth x share x monetization x take rate x margin x exit multiple. Baseline 1.0x = flat fundamentals sold at the entry-implied multiple P0 = entry x FDV supply / flat first-year EBITDA. When P0 is high the exit-multiple bar is a large give-back: the return is earnings-growth-driven, not re-rating-driven.

In [ ]:
# Per-case MOIC bridge; bars multiply to the case MOIC (asserted).
_wf = {c: model.moic_waterfall(c, scenario, ctx, marks_now, summary['returns_entry'])
       for c in CASES}
for _c, (_df, _m) in _wf.items():
    assert abs(_df['cum_moic'].iloc[-1] - val.loc[_c, 'moic']) < 1e-9 * max(1.0, val.loc[_c, 'moic'])

fig = display.moic_waterfall_chart({c: df for c, (df, _m) in _wf.items()},
                                   summary['returns_entry'],
                                   meta={c: m for c, (_df, m) in _wf.items()})

# factor table behind the bars
_ft = pd.DataFrame({c: [f'x{f:.2f}' for f in _wf[c][0]['factor']] for c in CASES},
                   index=list(_wf[CASES[0]][0]['step']))
_ft.loc['MOIC (product)'] = [f"{val.loc[c, 'moic']:.1f}x" for c in CASES]
print('MOIC bridge factors by case')
_ft

## Comps

TODO: state the comp-set rationale for this deal (why these peer groups; what business model the subject actually is). All figures TTM, everything at spot, levels and multiples on FDV (mcap alongside; equities' FDV is proxied by market cap, no EV).

FDV/EBITDA is the tie-out column for the model: it is the basis of `exit_pe` and of the section-4 entry multiple, and EBITDA Mgn anchors the model's margin ramp. TODO: justify the exit multiples against the comp medians (base near the relevant median; Bear as the de-rating case). Revenue is the NET take everywhere: total revenue for equities, DeFiLlama net protocol revenue for tokens + the subject (same altitude as the model's revenue line). Tokens do not report opex, so their EBITDA is blank and their read is FDV/Rev.

TODO: note any comp-specific caveats (consolidated business lines, fee switches off, etc.).

In [ ]:
# Subject row from live marks, at spot like every comp (the entry story lives in
# section 4). Rev = TTM net revenue; EBITDA blank (model assumption, not reported).
subject = dict(Company=f'{PROTOCOL_NAME} ({TICKER})', Group='SUBJECT', Price=PRICE_NOW,
               Mcap=marks_now['mcap'], Fdv=FDV_NOW,
               Rev=ttm_inc['revenue'] if ttm_inc else marks_now['revenue'],
               EBITDA=float('nan'))

print('Pulling comps (yfinance + DeFiLlama + CoinGecko)...')
comps_df = comps.build_comps(subject, EQUITY_COMPS, PEER_COMPS)
print('done.\n')
print('COMPS ($m) - all at spot; levels and multiples on FDV (equities: mcap as the FD proxy)')
print('   Revenue: total revenue for equities, TTM net protocol revenue for tokens + subject.')
print('   FDV/EBITDA is the tie-out to the model exit multiple; blank for tokens (opex unreported).\n')
print(display.comps_table(comps_df, subject_name=PROTOCOL_NAME).to_string())
print(f'\nGROUP MEDIANS (ex-{PROTOCOL_NAME})')
print(display.group_median_table(comps_df).to_string())

In [ ]:
# Cache each token comp's DeFiLlama income statement to data/financial_statements/{slug}.json
# (live pull overwrites; USE_LIVE_DATA=False reads the cache). Needs DEFILLAMA_API_KEY (Pro).
_fin_stmts = comps.fetch_comp_financial_statements(PEER_COMPS, use_live=USE_LIVE_DATA)
for _slug, _d in _fin_stmts.items():
    _yrs = sorted(_d['aggregates']['yearly'])
    print(f"  {_d.get('displayName', _slug):16s} {_yrs[0]}-{_yrs[-1]}  "
          f"-> data/financial_statements/{_slug}.json")

# Thesis & One-Pager

Reuses the valuation computed above, no recompute. Edit NARRATIVE and MANUAL, then this section writes `outputs/<slug>-one-pager.html` and previews it. For a new token: duplicate the notebook, change the knobs and narrative, Run All.

In [ ]:
import yaml
from valuation.pipeline import build_metrics
from onepager import save
from IPython.display import IFrame, display as ipy_display

## Narrative

In [ ]:
NARRATIVE = yaml.safe_load(r"""
protocol_name: <Protocol Name>            # EDIT everything in this block
ticker: <TICKER>
sector: <Sector, e.g. Institutional Credit / DEX / Infra>
verdict: HOLD           # BUY / ACCUMULATE / HOLD / REDUCE / SELL

# NOTE: any figures quoted in the thesis are hand-written; re-check them against
# the comps table above before sharing (they go stale).
thesis: >-
  TODO: 3-4 sentences. What you underwrite (the driver move that carries the
  return), the resulting revenue path, and the valuation anchor vs peers
  (premium/discount and why it is earned).

team:    'TODO: founders / key operators, one line.'
market:  'TODO: the TAM and why it grows, one line.'
product: 'TODO: what the protocol sells and how the token captures it, one line.'

overview: >-
  TODO: 3-5 sentences. What the protocol does, where revenue comes from, how
  value reaches the token (fee switch / buybacks / staking), and why users pick
  it over the alternatives.

management: >-
  TODO: 1-2 sentences on the team's background and execution record.

sentiment: >-
  TODO: 1-2 sentences on positioning, ownership, and how the market currently
  reads the token.

# catalysts & risks: write each line to convey WHEN it lands and WHY it matters for the token
catalysts:
  - 'TODO: <when it lands>; <what it is and why it moves the token>.'
  - 'TODO: <when it lands>; <what it is and why it moves the token>.'
  - 'TODO: <when it lands>; <what it is and why it moves the token>.'

risks:
  - 'TODO: <trigger/timing>; <the risk and any mitigant>.'
  - 'TODO: <trigger/timing>; <the risk and any mitigant>.'
  - 'TODO: <trigger/timing>; <the risk and any mitigant>.'

# token economics: fee_switch, token_rights and equity_capture are
# auto-pulled from the DeFiLlama token-rights dataset (see pipeline.build_metrics).

# footer
data_sources: DefiLlama and CoinGecko
""")

## Manual inputs

Position sizing is the only hand-entered input; everything else auto-fills from the valuation above. Use MANUAL only to override a specific figure (e.g. `traction_title`, `comps_note`, or a hand-set table).

In [ ]:
# Position sizing; allocation % = notional / fund size.
NOTIONAL_M  = None  # EDIT: $mm deployed; leave None until sized (renders as "n/a")
FUND_SIZE_M = 50    # $mm fund size

MANUAL = dict()     # hand-set overrides only; none needed by default

## Render & preview

In [ ]:
# build_metrics has no defaults: the notebook passes every cfg key it needs.
cfg = dict(base_case=ACTIVE_CASE, defillama_slug=DEFILLAMA_SLUG, gecko_id=GECKO_ID,
           annualize_window_days=ANNUALIZE_WINDOW_DAYS, peer_comps=PEER_COMPS,
           protocol_label=PROTOCOL_NAME, unit_label=UNIT_LABEL, units_metric=UNITS_METRIC)
valuation = dict(cfg=cfg, marks=marks_now, fdv_now=FDV_NOW, scenario=scenario,
                 ctx=ctx, val=val, summary=summary, meta=meta)

NUMBERS = build_metrics(valuation, comps_df=comps_df, manual=MANUAL,
                        notional_m=NOTIONAL_M, fund_size_m=FUND_SIZE_M)
assert NUMBERS['base_moic'] == f"{val.loc[ACTIVE_CASE, 'moic']:,.1f}x"   # ties to the notebook
context = {**NARRATIVE, **NUMBERS}
out = save(context, f"outputs/{DEFILLAMA_SLUG}-one-pager.html")
print("wrote", out)
ipy_display(IFrame(str(out), width=860, height=1120))